# 13 · How long the criterion remembers the distribution

**What this shows.** One number per mouse for chapter 3: how many trials the criterion averages over.
Two designs show it from two sides. In blocks of several days (pre-opto mice) the criterion moves towards
each new distribution; when the distribution switches every day (the opto mice's Hard phase) it does not
follow. One running average with a long memory produces both. Alongside it, the block shift with the slow
upward drift taken out: the Hard-B blocks came first, so drift alone would make Hard-A look higher.

**What produced it.** Two `timescale` runs:

```bash
sc-reports timescale --cohort behaviour1-cohort
sc-reports timescale --cohort opto1-cohort
```

each writing, under `timescale/<cohort>/<run_id>/`: `sessions.csv` (every session in order: criterion,
normative criterion, the fitted model), `fits.csv` (per mouse, variant and gain: the memory N with its 95 %
interval, offset, drift, gain), `shifts.csv` (Hard-A − Hard-B with the drift taken out), `profiles.csv`
(fit against N) and `timescale.pdf`. The model, the variants and the intervals are described at the top of
`sound_categorisation/behaviour/timescale.py`.

In [ ]:
from nb_setup import *
from sound_categorisation.plotting.timescale import plot_memory, plot_session_criterion

banner()
runs = {c: open_run('timescale', c) for c in dict.fromkeys([SWITCH_COHORT, OPTO_COHORT])}


def read(name):
    frames = [pd.read_csv(r / f'{name}.csv').assign(cohort=c) for c, r in runs.items() if (r / f'{name}.csv').exists()]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


sessions, fits, shifts = read('sessions'), read('fits'), read('shifts')
{k: v.shape for k, v in {'sessions': sessions, 'fits': fits, 'shifts': shifts}.items()}

## 1 · One mouse through the pipeline

`behaviour/timescale.py`: `session_sequence` lays out the sessions from the first Hard session to the last,
each with its criterion (`pse_fixed`: slope and lapses pinned to the mouse's expert-Uniform fit) and its
number of trials; `fit_integrator` runs the running average through them and finds the memory N.

In [ ]:
from sound_categorisation.behaviour.timescale import compute_timescale

exp = load_experiment()
blocked = sorted(fits.loc[fits['design'] == 'blocked', 'animal'].unique())
aid = blocked[0] if blocked else sorted(fits['animal'].unique())[0]
res = compute_timescale(exp.animals[aid])
print(res)
res.sessions[['order', 'distribution', 'session_type', 'n_trials', 'normative_pse', 'pse_all', 'predicted_main']].round(3)

## 2 · The figure

Left and middle: one mouse of each design, its criterion session by session (dots; open ones are not
fitted), the fitted running average (line) and the normative criterion (dashes). Right: the memory N per
mouse with its 95 % interval; an arrow means the data bound it only from below, because there is no swing
to fit. Open markers: mice the bias check flags (they stick to one side whatever the distribution).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), gridspec_kw={'width_ratios': [1, 1, 1.2]})
main_fixed = fits[(fits['variant'] == 'main') & (fits['gain_mode'] == 'fixed')]
for ax, design in zip(axes[:2], ('blocked', 'alternation')):
    ids = sorted(main_fixed.loc[(main_fixed['design'] == design) & ~main_fixed['biased_animal'].astype(bool), 'animal'])
    if not ids:
        ax.set_axis_off()
        continue
    plot_session_criterion(sessions[sessions['animal'] == ids[0]], 'main', ax=ax)
    f = main_fixed[main_fixed['animal'] == ids[0]].iloc[0]
    ax.set_title(f'{ids[0]} ({design}): N = {f.n_hat:.0f} [{f.n_low:.0f}, {f.n_high:.0f}]', fontsize=10)
axes[0].legend(fontsize=7, frameon=False)
plot_memory(fits, 'main', 'fixed', ax=axes[2])
axes[2].legend(fontsize=8, frameon=False)
axes[2].set_title('memory per mouse (main sessions, gain 1)', fontsize=10)
plt.tight_layout()

## 3 · The numbers

Per mouse: the memory N with its 95 % interval (main sessions, gain 1), the drift per session, and the
shift: Hard-A − Hard-B in session criterion with the drift taken out, as a share of the normative
difference (1 = all the way), with its 95 % interval.

In [ ]:
cols = ['animal', 'design', 'genotype', 'biased_animal', 'n_sessions', 'n_hat', 'n_low', 'n_high', 'drift']
table = main_fixed[cols].merge(
    shifts.loc[shifts['variant'] == 'main', ['animal', 'fraction', 'fraction_low', 'fraction_high']],
    on='animal', how='left')
table.sort_values(['design', 'animal']).round(3)

## 4 · Checks: the gain, and the laser days

**Gain.** With the gain fixed at 1 the criterion goes all the way to the normative criterion; a mouse that
shifts only part of the way would then be read as slow. The free-gain fit is the check on the blocked
mice (with the daily switch the gain cannot be told from N, so it says nothing there): if N barely
changes, the fixed-gain reading stands.

**Laser days.** For the opto mice the main fit uses only the six masking days. The two variants add the six
laser days (laser-off trials only; all trials), each with a constant for those days. Agreement means the
memory does not depend on the laser days. A difference would not be a PPC effect: light alone moves the
criterion, and the laser days came first.

In [ ]:
long = fits[['animal', 'design', 'variant', 'gain_mode', 'n_hat', 'n_low', 'n_high', 'gain']]
checks = pd.concat([long[(long['design'] == 'blocked') & (long['variant'] == 'main')],
                    long[(long['design'] == 'alternation') & (long['gain_mode'] == 'fixed')]])
checks.sort_values(['design', 'animal', 'variant', 'gain_mode']).round(2)

## 5 · Overnight

The model runs the sessions back to back. The switch report's `overnight.csv` (blocked mice) gives the
change in criterion from the last 100 trials of a session to the first 100 of the next, signed towards
the criterion before the switch (`toward_pre` > 0: drifting back overnight). Near zero, back to back is
fair; clearly positive, and the memory is partly in days rather than trials, which the chapter has to say.

In [ ]:
ov = load_switches(open_run('switch_adaptation', SWITCH_COHORT)).get('overnight', pd.DataFrame())
if len(ov):
    display(ov.groupby('animal').agg(pairs=('delta', 'size'), median_change=('delta', 'median'),
                                     median_toward_pre=('toward_pre', 'median')).round(3))
else:
    print('no overnight pairs in this run')